# Centauri Bloom — Unified trainer

One notebook for Colab, Kaggle, or a local computer (MacBook Air MPS, or Linux CUDA/CPU). In the settings cell choose the 14M continuation or the 50M from-scratch model. The existing 14M checkpoint, `configs/bloom-0.1.json`, and `Centauri-Bloom/Bloom-Text-0.1` are never overwritten. 50M runs reuse the same 8192-vocab tokenizer and publish to `Centauri-Bloom/Bloom-Text-0.2` as a reviewable PR. This notebook will not create that Hugging Face repo.

**Colab:** Runtime → T4 GPU. Outputs go to Google Drive (`MyDrive/centauri-bloom-runs`). Add a Colab secret named `HF_TOKEN`. Free sessions can disconnect, so a 50M run should set `RECOVER_RUN_DIR` to the existing Drive folder and re-run to resume from the last checkpoint.

**Kaggle:** Import this notebook, enable Internet and a GPU accelerator, and enable the `HF_TOKEN` secret. Attach the private contributor dataset only for Jaron. Outputs go to `/kaggle/working/centauri-bloom-runs`. Save a notebook version or download artifacts before the session ends. Resume with `RECOVER_RUN_DIR` the same way.

**Local:** Open this notebook from a checkout of the repo (Jupyter or VS Code). Checkpoints go to `./runs`. Export `HF_TOKEN` in the environment before the authorize cell. The trainer picks CUDA if present, otherwise MPS, otherwise CPU. A full 50M run on CPU will be very slow.

Candidate weights are submitted as a Hugging Face PR; review before merging. Never publish raw contributor data or secrets.

The final cell exports a matching experimental F32 GGUF and includes it alongside all existing candidate artifacts. No extra training is needed.


In [ ]:
#@title 1. Choose training settings
MODEL = "14M" #@param ["14M","50M"]
DATASET = "Jaron" #@param ["Jaron","FineWeb-Edu + TinyStories","FineWeb-Edu","TinyStories","Cosmopedia","Public-domain literature","Public-domain + TinyStories","FineWeb-Edu + literature + TinyStories","Arithmetic instruction pilot"]
STEPS = 3000 #@param [100,500,1000,1500,3000,5000,10000] {type:"raw"}
#@markdown Review the selected dataset's terms and confirm you may train on it and submit candidate weights.
CONFIRM_DATA_PERMISSION = False #@param {type:"boolean"}
#@markdown Optional: resume a previous unified run, including mid-run checkpoints, from this folder.
RECOVER_RUN_DIR = "" #@param {type:"string"}
JARON_FILE = "" #@param {type:"string"}
# Kaggle or local: optional exact path to the original private JSON; blank auto-detects on Colab/Kaggle.
if not CONFIRM_DATA_PERMISSION:
    raise ValueError("Confirm dataset permission before running.")
if MODEL not in ("14M", "50M"):
    raise ValueError("Choose 14M or 50M.")
if MODEL == "50M":
    print("50M trains from scratch and uploads to Centauri-Bloom/Bloom-Text-0.2.")
    print("It does not load or overwrite Bloom-Text-0.1. Recommended dataset: FineWeb-Edu + literature + TinyStories.")
print("Selected:", MODEL, "|", DATASET, "|", STEPS, "optimizer step attempts")


In [ ]:
#@title 2. Set up the environment
import os,sys,subprocess,json,shutil
from pathlib import Path
from datetime import datetime,timezone

def find_repo(start):
    cur = Path(start).resolve()
    for p in [cur, *cur.parents]:
        if (p/"pyproject.toml").is_file() and (p/"bloom"/"model.py").is_file():
            return p
    return None

IN_KAGGLE = Path("/kaggle/working").is_dir()
IN_COLAB = False
if not IN_KAGGLE:
    try:
        import google.colab  # noqa: F401
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
IN_LOCAL = not IN_KAGGLE and not IN_COLAB
os.environ["HF_HUB_DISABLE_XET"] = "1"

def checked(args):
    p = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout, flush=True)
    if p.returncode:
        raise RuntimeError(f"Setup failed (exit {p.returncode}). See full diagnostic above.")

if IN_KAGGLE:
    REPO = Path("/kaggle/working/centauri-bloom")
elif IN_COLAB:
    REPO = Path("/content/centauri-bloom")
else:
    REPO = find_repo(Path.cwd())
    if REPO is None:
        raise RuntimeError("Open this notebook from a centauri-bloom checkout, or run it from the repo root.")

if not IN_LOCAL:
    if not (REPO/".git").exists():
        checked(["git", "clone", "--depth", "1", "https://github.com/InterstellarNerd8760/centauri-bloom.git", str(REPO)])
    else:
        checked(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"])
os.chdir(REPO)
# Preserve Colab/Kaggle CUDA-enabled PyTorch and its matching dependencies.
checked([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."])
checked([sys.executable, "-m", "pip", "install", "tokenizers", "datasets", "huggingface_hub", "ipywidgets", "safetensors", "gguf"])

from bloom.notebook_workflow import submit_candidate, sha256, pick_device, preflight
from bloom.trainer import REPO_ID, TARGET_REPOS
from bloom.trainer_data import JARON_SHA, RECIPES

DEVICE = pick_device()
if not IN_LOCAL and DEVICE != "cuda":
    raise RuntimeError("Enable a GPU runtime (T4 on Colab, GPU accelerator on Kaggle).")
if IN_LOCAL and DEVICE == "cpu" and MODEL == "50M":
    print("Warning: a full 50M run on CPU will be very slow. MPS or CUDA is strongly preferred.")
if DEVICE == "cuda" and shutil.which("nvidia-smi"):
    checked(["nvidia-smi"])
preflight(DEVICE)
if DATASET not in RECIPES or type(STEPS) is not int or not 1 <= STEPS <= 10000:
    raise ValueError("Invalid dataset or step budget.")
if IN_KAGGLE:
    BASE = Path("/kaggle/working/centauri-bloom-runs")
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/centauri-bloom-runs")
else:
    BASE = REPO / "runs"
BASE.mkdir(parents=True, exist_ok=True)
slug = (MODEL + "-" + DATASET).lower().replace(" + ", "-").replace(" ", "-")
OUT_DIR = Path(RECOVER_RUN_DIR) if RECOVER_RUN_DIR else BASE / ("bloom-" + slug + "-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
HF_REPO_ID = TARGET_REPOS[MODEL]
print("Environment:", "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local", "| device:", DEVICE, "| output:", OUT_DIR)
print("Upload target:", HF_REPO_ID)


In [ ]:
#@title 3. Authorize Hugging Face
from getpass import getpass
from bloom.notebook_workflow import auth_client
try:
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    elif IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    else:
        token = os.environ.get("HF_TOKEN")
except Exception:
    token = None
if not token:
    token = getpass("Hugging Face write token: ")
AUTH_API = auth_client(HF_REPO_ID, token, require_repo=(MODEL == "14M"))
del token
print("Authenticated. Candidate PR only. Target:", HF_REPO_ID)


In [ ]:
#@title 4. Load the selected model, prepare data, train and evaluate
if AUTH_API is None:
    raise RuntimeError("Tap Authorize upload in the preceding cell.")
jaron_path = None
if DATASET == "Jaron" and not RECOVER_RUN_DIR:
    if JARON_FILE:
        jaron_path = Path(JARON_FILE)
    elif IN_KAGGLE:
        matches = list(Path("/kaggle/input").rglob("bloom_instruction_expanded.json"))
        matches = [p for p in matches if sha256(p) == JARON_SHA]
        if len(matches) != 1:
            raise ValueError("Attach the original private contributor dataset or set JARON_FILE to its exact path.")
        jaron_path = matches[0]
    elif IN_COLAB:
        folder = Path("/content/drive/MyDrive/centauri-bloom-data")
        folder.mkdir(parents=True, exist_ok=True)
        jaron_path = folder / "bloom_instruction_expanded.json"
        if not jaron_path.exists():
            from google.colab import files
            uploaded = files.upload()
            raw = uploaded.get(jaron_path.name)
            if raw is None:
                raise ValueError("Select the original contributor JSON.")
            import hashlib
            if hashlib.sha256(raw).hexdigest() != JARON_SHA:
                raise ValueError("Contributor file hash mismatch.")
            jaron_path.write_bytes(raw)
    else:
        default = Path.home() / "Downloads" / "bloom_instruction_expanded.json"
        jaron_path = default if default.is_file() else None
        if jaron_path is None:
            raise ValueError("Set JARON_FILE to the original contributor JSON, or place it in Downloads.")
    if sha256(jaron_path) != JARON_SHA:
        raise ValueError("Contributor file hash mismatch.")
# Train in a fresh process, avoiding stale notebook imports and Dynamo state.
env = os.environ.copy()
env["HF_TOKEN"] = AUTH_API.token
payload = json.dumps(dict(recipe=DATASET, steps=STEPS, repo_dir=str(REPO), out_dir=str(OUT_DIR),
                          jaron_path=str(jaron_path) if jaron_path else None,
                          recover=bool(RECOVER_RUN_DIR), model_size=MODEL, device=DEVICE))
script = ("import json,sys,os; from huggingface_hub import HfApi; from bloom.notebook_workflow import preflight; "
          "from bloom.trainer import run_training; args=json.loads(sys.argv[1]); preflight(args['device']); "
          "run_training(HfApi(token=os.environ['HF_TOKEN']), **args)")
try:
    result = subprocess.run([sys.executable, "-u", "-c", script, payload], env=env)
finally:
    env.pop("HF_TOKEN", None)
if result.returncode:
    raise RuntimeError("Training failed. See diagnostic above; saved files remain in the output directory. Set RECOVER_RUN_DIR to that folder to resume.")
metrics = json.loads((OUT_DIR / "metrics.json").read_text())


In [ ]:
#@title 5. Export GGUF and automatically submit the completed candidate
if AUTH_API is None:
    raise RuntimeError("Authorize upload first.")
metrics = json.loads((OUT_DIR / "metrics.json").read_text())
if metrics["additional_steps"] != STEPS or metrics["dataset"] != DATASET or metrics.get("model_size", "14M") != MODEL:
    raise RuntimeError("Run settings differ from completed candidate.")
if MODEL == "50M" and HF_REPO_ID != "Centauri-Bloom/Bloom-Text-0.2":
    raise RuntimeError("50M candidates must target Bloom-Text-0.2, not Bloom-Text-0.1.")
submit_candidate(AUTH_API, HF_REPO_ID, OUT_DIR, STEPS, metrics["parent_revision"], metrics["run_name"])
print("Review and merge the candidate in Hugging Face before the next run.")
if MODEL == "50M":
    print("If the Bloom-Text-0.2 repo does not exist yet, create it on Hugging Face, then retry this cell or use upload recovery. This notebook will not create the repo.")
